In [0]:
%sql
USE CATALOG dbacademy;
USE SCHEMA labuser14586003_1778453479;

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/dbacademy/labuser14586003_1778453479/data/raw/sub/orders_01.csv")


In [0]:
display(df)

In [0]:
df.write.mode("overwrite")\
    .format("json")\
    .save("/Volumes/dbacademy/labuser14586003_1778453479/data/raw/orders")

In [0]:
dbutils.fs.ls('/Volumes/dbacademy/labuser14586003_1778453479/data/raw/')

In [0]:
df_main = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/dbacademy/labuser14586003_1778453479/data/raw/")
display(df_main)

In [0]:
spark.read.format('csv')\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .option("pathGlobFilter", "*.csv")\
    .option("recursiveFileLookup", "true")\
    .load("/Volumes/dbacademy/labuser14586003_1778453479/data/raw/")\
    .display()

In [0]:
df_auto = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option('cloudFiles.inferColumnTypes', 'true')\
    .option("cloudFiles.maxFilesPerTrigger", 1)\
    .option("cloudFiles.schemaEvolutionMode", "rescue")\
.option("cloudFiles.schemaLocation","/Volumes/dbacademy/labuser14586003_1778453479/data/checkpoint")\
    .load("/Volumes/dbacademy/labuser14586003_1778453479/data/raw/")

In [0]:
df_auto.writeStream\
    .option("checkpointLocation", "/Volumes/dbacademy/labuser14586003_1778453479/data/checkpoint")\
    .toTable("orders")

In [0]:
%sql
DROP TABLE IF EXISTS orders;


In [0]:
df = spark.read.table("orders")
display(df)

In [0]:
schema = schema_of_json(df.select("_rescued_data").collect()[0][0])
schema

In [0]:
df = df.withColumn("rescued_data", from_json("_rescued_data", schema))
display(df)

In [0]:
df.withColumn("price", col("rescued_data.price").cast(DoubleType()))\
    .withColumn('quantity', col("rescued_data.quantity").cast(IntegerType()))\
    .drop('_rescued_data', 'rescued_data')\
    .display()
